## 1、如何去构建一个tools

In [18]:
from langchain_core.tools import tool


## 方式1：通过装饰器的方式去定义。在函数的前面上面添加@tool装饰器
@tool
def calculate_expo(base: int, expo: int):
    """
    一个用于计算幂的函数
    :param base:
    :param expo:
    :return:
    """
    return base ** expo


print(type(calculate_expo))
calculate_expo

<class 'langchain_core.tools.structured.StructuredTool'>


StructuredTool(name='calculate_expo', description='一个用于计算幂的函数\n:param base:\n:param expo:\n:return:', args_schema=<class 'langchain_core.utils.pydantic.calculate_expo'>, func=<function calculate_expo at 0x000002718A72E020>)

In [19]:
print(calculate_expo.name)
print(calculate_expo.description)
print(calculate_expo.args)

calculate_expo
一个用于计算幂的函数
:param base:
:param expo:
:return:
{'base': {'title': 'Base', 'type': 'integer'}, 'expo': {'title': 'Expo', 'type': 'integer'}}


In [20]:
## 方式2：通过调用tool函数去定义

def calculate_expo2(base: int, expo: int):
    """
    一个用于计算幂的函数
    :param base:
    :param expo:
    :return:
    """
    return base ** expo


calculate_expo2 = tool(calculate_expo2)
calculate_expo2

StructuredTool(name='calculate_expo2', description='一个用于计算幂的函数\n:param base:\n:param expo:\n:return:', args_schema=<class 'langchain_core.utils.pydantic.calculate_expo2'>, func=<function calculate_expo2 at 0x00000271C28B7EC0>)

In [21]:
# ## 定义函数入参结构时，除了使用typed hint(也就是base:int这种类型推断)外，我们还可以使用我们的pydantic
from pydantic import BaseModel, Field


def calculate_expo3(base, expo):
    return base ** expo


class CalcExpoSchema(BaseModel):
    base: int = Field(description="幂的底数")
    expo: int = Field(description="幂的指数")


calculate_expo3 = tool(calculate_expo3, args_schema=CalcExpoSchema, description="一个用于计算幂的函数")
print(calculate_expo3.args)
print(type(calculate_expo2))
print(type(calculate_expo3))
# print(type(a_new_tool))

{'base': {'description': '幂的底数', 'title': 'Base', 'type': 'integer'}, 'expo': {'description': '幂的指数', 'title': 'Expo', 'type': 'integer'}}
<class 'langchain_core.tools.structured.StructuredTool'>
<class 'langchain_core.tools.structured.StructuredTool'>


In [22]:
args = {"base": 2, "expo": 3}
globals()['calculate_expo2'].invoke(args)

8

## 2、如何去调用tool
通过调用invoke去执行tool当中的逻辑，invoke传入一个dict，dict的键就是tool的入参，值就是参数的值

In [23]:
calculate_expo3.invoke({"base": "2", "expo": 3})

## 3、大模型如何感知到有哪些tools

In [24]:
from langchain_openai import ChatOpenAI
import dotenv

dotenv.load_dotenv()
llm = ChatOpenAI(
    model="gpt-4o-mini"
)

In [25]:
import logging

logging.basicConfig(level=logging.DEBUG, format="%(asctime)s %(filename)s %(message)s")

In [26]:
llm.invoke("你好")

2026-04-11 16:16:30,160 _base_client.py Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-c686a041-7496-4593-a194-3258b9ef27a5', 'content': None, 'json_data': {'messages': [{'content': '你好', 'role': 'user'}], 'model': 'gpt-4o-mini', 'stream': False}}
2026-04-11 16:16:30,161 _base_client.py Sending HTTP Request: POST https://api.openai-proxy.org/v1/chat/completions
2026-04-11 16:16:30,162 _trace.py close.started
2026-04-11 16:16:30,162 _trace.py close.complete
2026-04-11 16:16:30,163 _trace.py connect_tcp.started host='api.openai-proxy.org' port=443 local_address=None timeout=None socket_options=None
2026-04-11 16:16:30,391 _trace.py connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x00000271C2784510>
2026-04-11 16:16:30,392 _trace.py start_tls.started ssl_context=<ssl.SSLContext object at 0x00000271C21AA2A0> server_hostname='api.openai-

AIMessage(content='你好！有什么我可以帮助你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 8, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DTNo1HB2LoTEJUUJVE95YAh6FLqf2', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d7b9d-27d0-7671-bfb0-d1d4273fa23b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 10, 'total_tokens': 18, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [27]:
tools = [calculate_expo3]
llm_with_tools_bound = llm.bind_tools(tools)

In [28]:
res = llm_with_tools_bound.invoke("帮我计算一下2的10次方是多少")

2026-04-11 16:16:33,102 _base_client.py Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-8a3e15b5-0e2e-4b7c-b5a1-b578f801e33f', 'content': None, 'json_data': {'messages': [{'content': '帮我计算一下2的10次方是多少', 'role': 'user'}], 'model': 'gpt-4o-mini', 'stream': False, 'tools': [{'type': 'function', 'function': {'name': 'calculate_expo3', 'description': '一个用于计算幂的函数', 'parameters': {'properties': {'base': {'description': '幂的底数', 'type': 'integer'}, 'expo': {'description': '幂的指数', 'type': 'integer'}}, 'required': ['base', 'expo'], 'type': 'object'}}}]}}
2026-04-11 16:16:33,102 _base_client.py Sending HTTP Request: POST https://api.openai-proxy.org/v1/chat/completions
2026-04-11 16:16:33,103 _trace.py send_request_headers.started request=<Request [b'POST']>
2026-04-11 16:16:33,103 _trace.py send_request_headers.complete
2026-04-11 16:16:33,104 _trace.py send_request_body.start

In [29]:
res  # 此时AIMessage当中的content没有内容。当前AI无法自动去调用工具，需要我们手动去调用

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 73, 'total_tokens': 94, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DTNo2mm0GGFGLW8RR8dcUHF3vfBpb', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d7b9d-334e-7ef3-baf4-0ef01cb2163d-0', tool_calls=[{'name': 'calculate_expo3', 'args': {'base': 2, 'expo': 10}, 'id': 'call_cYkEtKjbDsXTaCw1iQgHoAQt', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 73, 'output_tokens': 21, 'total_tokens': 94, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [30]:
# str2tool={
#     "calculate_expo3":pydantic_tool,
#     # 维护其他tool_name和真正的tool实例之间的映射
# }
for tool_call in res.tool_calls:
    print(tool_call)
    tool_name = tool_call['name']
    args = tool_call['args']
    tool_invoke_res = globals()[tool_name](base=2, expo=10)
    print(tool_invoke_res)


{'name': 'calculate_expo3', 'args': {'base': 2, 'expo': 10}, 'id': 'call_cYkEtKjbDsXTaCw1iQgHoAQt', 'type': 'tool_call'}


TypeError: 'StructuredTool' object is not callable

In [ ]:
globals()["calc_expo"]

## 4、tools的底层原理
利用大模型厂商所提供的工具调用的能力。